## Cell 1: Imports, Config, Utility Functions

In [2]:
import os
import sys
import time
import math
import random
import numpy as np
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Union, Any
from collections import OrderedDict, defaultdict
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split, WeightedRandomSampler
from torch.utils.tensorboard import SummaryWriter
from torch.cuda.amp import GradScaler, autocast
import torchvision.transforms as transforms
from torchvision.transforms import functional as TF
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, roc_curve, auc, classification_report,
    cohen_kappa_score, balanced_accuracy_score,
    matthews_corrcoef, average_precision_score,
    precision_recall_curve
)

from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

try:
    from thop import profile, clever_format
    THOP_AVAILABLE = True
except ImportError:
    THOP_AVAILABLE = False
    print("thop not installed. FLOPs calculation will be skipped.")

# ============================================================================
# Dynamic Configuration (Multi-Dataset)
# ============================================================================
class Config:
    # ---- Choose dataset here ----
    dataset_name = "NWPU"   # Options: "NWPU", "EuroSAT", "PatternNet", "MLRSNet", "AID"

    # ---- Registry of prepared dataset roots (CHANGE THESE PATHS TO YOUR ACTUAL LOCATIONS) ----
    dataset_roots = {
        "NWPU":      r"C:\Users\aipmu\Videos\Lesss go\Datasets_Prepared\NWPU",
        "EuroSAT":   r"C:\Users\aipmu\Videos\Lesss go\Datasets_Prepared\EuroSAT",
        "PatternNet":r"C:\Users\aipmu\Videos\Lesss go\Datasets_Prepared\PatternNet",
        "MLRSNet":   r"C:\Users\aipmu\Videos\Lesss go\Datasets_Prepared\MLRSNet",
        "AID":       r"C:\Users\aipmu\Videos\Lesss go\Datasets_Prepared\AID",
    }
    num_classes_dict = {"NWPU":45, "EuroSAT":10, "PatternNet":38, "MLRSNet":46, "AID":30}

    @property
    def dataset_root(self):
        return self.dataset_roots[self.dataset_name]

    @property
    def num_classes(self):
        return self.num_classes_dict[self.dataset_name]

    # ---- Model hyperparameters (unchanged from your original) ----
    input_size = 256
    init_channels = 16
    batch_size = 64
    num_workers = 0
    epochs = 150
    lr = 0.001
    weight_decay = 0.01
    betas = (0.9, 0.999)
    warmup_epochs = 5
    min_lr = 1e-6
    label_smoothing = 0.1
    mixup_alpha = 0.2
    cutmix_alpha = 0.2
    drop_path_rate = 0.1
    dropout_rate = 0.3
    seed = 42
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    save_dir = 'nwpu_resisc45_checkpoints'
    early_stop_patience = 20

    @classmethod
    def print_config(cls):
        print("\n" + "="*60)
        print("CONFIGURATION")
        print("="*60)
        for key, value in cls.__dict__.items():
            if not key.startswith('__') and not callable(value):
                print(f"{key}: {value}")
        print("="*60 + "\n")

# ============================================================================
# Utility Functions
# ============================================================================
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def init_weights(module: nn.Module):
    if isinstance(module, (nn.Conv2d, nn.Linear)):
        nn.init.kaiming_normal_(module.weight, mode='fan_out', nonlinearity='relu')
        if module.bias is not None:
            nn.init.constant_(module.bias, 0)
    elif isinstance(module, nn.BatchNorm2d):
        nn.init.constant_(module.weight, 1)
        nn.init.constant_(module.bias, 0)

print("Cell 1 done.")

RuntimeError: CPU dispatcher tracer already initlized

## Cell 2: Dataset Class (Generic for any folder structure)

In [7]:
class NWPURESISC45Dataset(Dataset):
    """Custom dataset that works for any dataset with train/ and test/ folders."""
    def __init__(self, root_dir: str, split: str = 'train', transform=None):
        self.root_dir = Path(root_dir)
        self.split = split
        self.transform = transform

        if split == 'train':
            self.split_dir = self.root_dir / 'train'
        else:
            self.split_dir = self.root_dir / 'test'

        self.classes = sorted([d.name for d in self.split_dir.iterdir() if d.is_dir()])
        self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(self.classes)}

        self.images = []
        self.labels = []
        for class_name in self.classes:
            class_dir = self.split_dir / class_name
            class_idx = self.class_to_idx[class_name]
            for ext in ['*.jpg', '*.jpeg', '*.png', '*.tif', '*.tiff']:
                for img_path in class_dir.glob(ext):
                    self.images.append(img_path)
                    self.labels.append(class_idx)
        print(f"Loaded {split} split: {len(self.images)} images, {len(self.classes)} classes")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        label = self.labels[idx]
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            print(f"Error loading {img_path}: {e}")
            return self.__getitem__(random.randint(0, len(self)-1))
        if self.transform:
            image = self.transform(image)
        return image, label

## Cell 3: Data Augmentation Classes (MixUp, CutMix) and Transforms

In [8]:
class MixUp:
    def __init__(self, alpha=0.2):
        self.alpha = alpha
    def __call__(self, images, targets):
        if self.alpha > 0:
            lam = np.random.beta(self.alpha, self.alpha)
        else:
            lam = 1
        batch_size = images.size(0)
        index = torch.randperm(batch_size).to(images.device)
        mixed_images = lam * images + (1 - lam) * images[index]
        targets_a, targets_b = targets, targets[index]
        return mixed_images, targets_a, targets_b, lam

class CutMix:
    def __init__(self, alpha=0.2):
        self.alpha = alpha
    def __call__(self, images, targets):
        if self.alpha > 0:
            lam = np.random.beta(self.alpha, self.alpha)
        else:
            lam = 1
        batch_size = images.size(0)
        index = torch.randperm(batch_size).to(images.device)
        bbx1, bby1, bbx2, bby2 = self.rand_bbox(images.size(), lam)
        images[:, :, bbx1:bbx2, bby1:bby2] = images[index, :, bbx1:bbx2, bby1:bby2]
        lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (images.size(-1) * images.size(-2)))
        return images, targets, targets[index], lam
    def rand_bbox(self, size, lam):
        W = size[2]; H = size[3]
        cut_rat = np.sqrt(1. - lam)
        cut_w = int(W * cut_rat); cut_h = int(H * cut_rat)
        cx = np.random.randint(W); cy = np.random.randint(H)
        bbx1 = np.clip(cx - cut_w // 2, 0, W)
        bby1 = np.clip(cy - cut_h // 2, 0, H)
        bbx2 = np.clip(cx + cut_w // 2, 0, W)
        bby2 = np.clip(cy + cut_h // 2, 0, H)
        return bbx1, bby1, bbx2, bby2

def get_train_transforms(input_size=256):
    return transforms.Compose([
        transforms.Resize(int(input_size * 1.125)),
        transforms.RandomResizedCrop(input_size, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.3),
        transforms.RandomRotation(30),
        transforms.RandomAffine(0, translate=(0.1,0.1), scale=(0.8,1.2), shear=10),
        transforms.ColorJitter(0.3,0.3,0.3,0.15),
        transforms.ToTensor(),
        transforms.RandomErasing(p=0.25, scale=(0.02,0.2)),
    ])

def get_test_transforms(input_size=256):
    return transforms.Compose([
        transforms.Resize(int(input_size * 1.125)),
        transforms.CenterCrop(input_size),
        transforms.ToTensor(),
    ])

## Cell 4: Model Components (Attention, ASPP, StochasticDepth, SEBlock, ResidualBlock)

In [9]:
class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(channels, channels // reduction, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // reduction, channels, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        return x * self.sigmoid(avg_out + max_out)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x_cat = torch.cat([avg_out, max_out], dim=1)
        out = self.conv(x_cat)
        return x * self.sigmoid(out)

class CBAM(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.channel_attention = ChannelAttention(channels, reduction)
        self.spatial_attention = SpatialAttention()
    def forward(self, x):
        x = self.channel_attention(x)
        x = self.spatial_attention(x)
        return x

class LightweightSelfAttention(nn.Module):
    def __init__(self, in_channels, reduction=8, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = in_channels // num_heads
        assert self.head_dim * num_heads == in_channels, "in_channels must be divisible by num_heads"
        self.query = nn.Conv2d(in_channels, in_channels, 1)
        self.key = nn.Conv2d(in_channels, in_channels, 1)
        self.value = nn.Conv2d(in_channels, in_channels, 1)
        self.gamma = nn.Parameter(torch.zeros(1))
    def forward(self, x):
        B, C, H, W = x.size()
        Q = self.query(x).view(B, self.num_heads, self.head_dim, H*W)
        K = self.key(x).view(B, self.num_heads, self.head_dim, H*W)
        V = self.value(x).view(B, self.num_heads, self.head_dim, H*W)
        Q = Q.transpose(-1, -2)
        energy = torch.matmul(Q, K) / (self.head_dim ** 0.5)
        attention = F.softmax(energy, dim=-1)
        out = torch.matmul(attention, V.transpose(-1, -2))
        out = out.transpose(-1, -2).contiguous().view(B, C, H, W)
        return self.gamma * out + x

class ASPP(nn.Module):
    def __init__(self, in_channels, out_channels, rates=[1,6,12,18]):
        super().__init__()
        self.aspp_blocks = nn.ModuleList()
        for rate in rates:
            self.aspp_blocks.append(
                nn.Sequential(
                    nn.Conv2d(in_channels, out_channels, 3, padding=rate, dilation=rate, bias=False),
                    nn.BatchNorm2d(out_channels),
                    nn.ReLU(inplace=True)
                )
            )
        self.global_avg_pool = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
        self.concat_proj = nn.Sequential(
            nn.Conv2d(out_channels * (len(rates)+1), out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        outs = [block(x) for block in self.aspp_blocks]
        global_feat = self.global_avg_pool(x)
        global_feat = F.interpolate(global_feat, size=x.shape[2:], mode='bilinear', align_corners=True)
        outs.append(global_feat)
        out = self.concat_proj(torch.cat(outs, dim=1))
        return out

class StochasticDepth(nn.Module):
    def __init__(self, drop_prob=0.0):
        super().__init__()
        self.drop_prob = drop_prob
    def forward(self, x):
        if not self.training or self.drop_prob == 0:
            return x
        keep_prob = 1 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        random_tensor = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
        random_tensor.floor_()
        return x.div(keep_prob) * random_tensor

class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, channels // reduction, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // reduction, channels, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return x * self.fc(x)

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, drop_path_prob=0.0,
                 use_se=False, use_cbam=False):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_channels)
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False)

        self.attention = nn.Identity()
        if use_se:
            self.attention = SEBlock(out_channels)
        elif use_cbam:
            self.attention = CBAM(out_channels)

        self.drop_path = StochasticDepth(drop_path_prob)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
    def forward(self, x):
        identity = x
        out = F.relu(self.bn1(x))
        out = self.conv1(out)
        out = F.relu(self.bn2(out))
        out = self.conv2(out)
        out = self.attention(out)
        out = self.drop_path(out)
        out += self.shortcut(identity)
        return out

## Cell 5: Main Model with Ablation Flags

In [10]:
class RemoteSensingHybridCNN(nn.Module):
    def __init__(self, num_classes=45, init_channels=16, drop_path_rate=0.1, dropout_rate=0.3,
                 use_se=True, use_cbam=True, use_self_attn=True, use_aspp=True):
        super().__init__()
        self.use_se = use_se
        self.use_cbam = use_cbam
        self.use_self_attn = use_self_attn
        self.use_aspp = use_aspp

        # Stem
        self.stem = nn.Sequential(
            nn.Conv2d(3, init_channels, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(init_channels), nn.ReLU(inplace=True),
            nn.Conv2d(init_channels, init_channels*2, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(init_channels*2), nn.ReLU(inplace=True),
            nn.MaxPool2d(3, stride=2, padding=1)
        )

        total_blocks = 16
        block_idx = 0

        # Stage 1 (no attention)
        self.stage1 = self._make_stage(init_channels*2, init_channels*4, 2, 1,
                                       drop_path_rate, block_idx, total_blocks,
                                       use_se=False, use_cbam=False)
        block_idx += 2

        # Stage 2: SE only if not using CBAM (avoid double attention)
        self.stage2 = self._make_stage(init_channels*4, init_channels*8, 3, 2,
                                       drop_path_rate, block_idx, total_blocks,
                                       use_se=self.use_se and not self.use_cbam, use_cbam=False)
        block_idx += 3

        # ASPP (optional)
        if self.use_aspp:
            self.aspp = nn.ModuleList([
                nn.Sequential(
                    nn.Conv2d(init_channels*8, init_channels*4, 3, padding=r, dilation=r, bias=False),
                    nn.BatchNorm2d(init_channels*4), nn.ReLU(inplace=True)
                ) for r in [1,3]
            ])
            self.aspp_fusion = nn.Sequential(
                nn.Conv2d(init_channels*4*len(self.aspp), init_channels*8, 1, bias=False),
                nn.BatchNorm2d(init_channels*8), nn.ReLU(inplace=True)
            )
        else:
            self.aspp = self.aspp_fusion = None

        # Stage 3: CBAM if enabled
        self.stage3 = self._make_stage(init_channels*8, init_channels*16, 4, 2,
                                       drop_path_rate, block_idx, total_blocks,
                                       use_se=self.use_se and not self.use_cbam, use_cbam=self.use_cbam)
        block_idx += 4
        
        # Self-attention (optional)
        if self.use_self_attn:
            self.self_attn = nn.Sequential(
                nn.Conv2d(init_channels*16, init_channels*2, 1), nn.ReLU(inplace=True),
                nn.Conv2d(init_channels*2, init_channels*16, 1), nn.Sigmoid()
            )
        else:
            self.self_attn = None

        # Stage 4
        self.stage4 = self._make_stage(init_channels*16, init_channels*32, 2, 2,
                                       drop_path_rate, block_idx, total_blocks,
                                       use_se=False, use_cbam=False)

        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(init_channels*32, 256),
            nn.BatchNorm1d(256), nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate*0.5),
            nn.Linear(256, num_classes)
        )
        self.apply(init_weights)
        nn.init.xavier_uniform_(self.classifier[-1].weight, gain=0.01)

    def _make_stage(self, in_ch, out_ch, num_blocks, stride, drop_path_rate,
                    block_start, total_blocks, use_se=False, use_cbam=False):
        blocks = []
        for i in range(num_blocks):
            drop_prob = drop_path_rate * (block_start + i) / (total_blocks - 1)
            if i == 0:
                blocks.append(ResidualBlock(in_ch, out_ch, stride, drop_prob, use_se, False))
            else:
                blocks.append(ResidualBlock(out_ch, out_ch, 1, drop_prob, use_se, False))
        return nn.Sequential(*blocks)

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)

        if self.use_aspp and self.aspp is not None:
            aspp_outs = [block(x) for block in self.aspp]
            x_aspp = torch.cat(aspp_outs, dim=1)
            x = x + self.aspp_fusion(x_aspp)

        x = self.stage3(x)

        if self.use_self_attn and self.self_attn is not None:
            attn = self.self_attn(x)
            x = x * attn

        x = self.stage4(x)
        x = self.global_pool(x).view(x.size(0), -1)
        return self.classifier(x)

## Cell 6: Loss Functions and Scheduler

In [11]:
class LabelSmoothingCrossEntropy(nn.Module):
    def __init__(self, smoothing=0.1):
        super().__init__()
        self.smoothing = smoothing
    def forward(self, pred, target):
        log_probs = F.log_softmax(pred, dim=-1)
        n_classes = pred.size(-1)
        with torch.no_grad():
            true_dist = torch.zeros_like(log_probs)
            true_dist.fill_(self.smoothing / (n_classes - 1))
            true_dist.scatter_(1, target.unsqueeze(1), 1.0 - self.smoothing)
        return torch.mean(torch.sum(-true_dist * log_probs, dim=-1))

class CombinedLoss(nn.Module):
    def __init__(self, smoothing=0.1):
        super().__init__()
        self.criterion = LabelSmoothingCrossEntropy(smoothing)
    def forward(self, pred, target):
        return self.criterion(pred, target)

class WarmupCosineScheduler:
    def __init__(self, optimizer, warmup_epochs, total_epochs, min_lr=1e-6, restart_epochs=None):
        self.optimizer = optimizer
        self.warmup_epochs = warmup_epochs
        self.total_epochs = total_epochs
        self.min_lr = min_lr
        self.restart_epochs = restart_epochs or []
        self.base_lrs = [group['lr'] for group in optimizer.param_groups]
        self.current_restart = 0
    def step(self, epoch):
        if epoch in self.restart_epochs:
            self.current_restart = epoch
            for group, base_lr in zip(self.optimizer.param_groups, self.base_lrs):
                group['lr'] = base_lr
        if epoch < self.warmup_epochs:
            alpha = epoch / self.warmup_epochs
            for group, base_lr in zip(self.optimizer.param_groups, self.base_lrs):
                group['lr'] = base_lr * alpha
        else:
            progress = (epoch - max(self.warmup_epochs, self.current_restart)) / \
                       (self.total_epochs - max(self.warmup_epochs, self.current_restart))
            progress = min(progress, 1.0)
            for group, base_lr in zip(self.optimizer.param_groups, self.base_lrs):
                group['lr'] = self.min_lr + (base_lr - self.min_lr) * (1 + math.cos(math.pi * progress)) / 2

## Cell 7: Trainer Class (Enhanced)

In [12]:
class Trainer:
    def __init__(self, model, train_loader, val_loader, test_loader, num_classes, device, save_dir, config):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.test_loader = test_loader
        self.num_classes = num_classes
        self.device = device
        self.save_dir = Path(save_dir)
        self.save_dir.mkdir(exist_ok=True)
        self.config = config

        self.criterion = CombinedLoss(smoothing=config.label_smoothing)
        self.optimizer = self._create_optimizer()
        self.scheduler = WarmupCosineScheduler(
            self.optimizer, config.warmup_epochs, config.epochs, config.min_lr,
            restart_epochs=[100,150]
        )
        self.scaler = GradScaler()
        self.mixup = MixUp(alpha=config.mixup_alpha)
        self.cutmix = CutMix(alpha=config.cutmix_alpha)

        self.current_epoch = 0
        self.best_val_acc = 0.0
        self.best_val_loss = float('inf')
        self.patience_counter = 0
        self.writer = SummaryWriter(log_dir=self.save_dir / 'logs')
        self.model = model.to(device)

        self.train_losses, self.val_losses = [], []
        self.train_accs, self.val_accs = [], []

        print(f"Model has {count_parameters(model):,} trainable parameters")

    def _create_optimizer(self):
        decay_params, no_decay_params = [], []
        for name, param in self.model.named_parameters():
            if not param.requires_grad:
                continue
            if len(param.shape) == 1 or name.endswith(".bias"):
                no_decay_params.append(param)
            else:
                decay_params.append(param)
        return torch.optim.AdamW([
            {'params': decay_params, 'weight_decay': self.config.weight_decay},
            {'params': no_decay_params, 'weight_decay': 0.0}
        ], lr=self.config.lr, betas=self.config.betas)

    def train_epoch(self):
        self.model.train()
        total_loss, correct, total = 0, 0, 0
        pbar = tqdm(self.train_loader, desc=f'Epoch {self.current_epoch}')
        for batch_idx, (images, targets) in enumerate(pbar):
            images, targets = images.to(self.device), targets.to(self.device)
            # MixUp/CutMix
            use_mixup = random.random() < 0.5
            use_cutmix = not use_mixup and random.random() < 0.5
            if use_mixup and self.config.mixup_alpha > 0:
                images, targets_a, targets_b, lam = self.mixup(images, targets)
            elif use_cutmix and self.config.cutmix_alpha > 0:
                images, targets_a, targets_b, lam = self.cutmix(images, targets)
            else:
                targets_a = targets_b = None
                lam = 1.0

            self.optimizer.zero_grad()
            with autocast():
                outputs = self.model(images)
                if targets_a is not None:
                    loss = lam * self.criterion(outputs, targets_a) + (1-lam) * self.criterion(outputs, targets_b)
                else:
                    loss = self.criterion(outputs, targets)

            self.scaler.scale(loss).backward()
            self.scaler.unscale_(self.optimizer)
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            self.scaler.step(self.optimizer)
            self.scaler.update()

            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            if targets_a is not None:
                correct += (lam * predicted.eq(targets_a).sum().item() +
                           (1-lam) * predicted.eq(targets_b).sum().item())
            else:
                correct += predicted.eq(targets).sum().item()

            pbar.set_postfix({'loss': total_loss/(batch_idx+1), 'acc': 100.*correct/total})

        return {'loss': total_loss/len(self.train_loader), 'accuracy': 100.*correct/total}

    def validate(self, loader):
        self.model.eval()
        total_loss, correct, total = 0, 0, 0
        all_preds, all_targets, all_probs = [], [], []
        with torch.no_grad():
            for images, targets in tqdm(loader, desc='Validation'):
                images, targets = images.to(self.device), targets.to(self.device)
                with autocast():
                    outputs = self.model(images)
                    loss = self.criterion(outputs, targets)
                total_loss += loss.item()
                probs = F.softmax(outputs, dim=1)
                _, predicted = outputs.max(1)
                total += targets.size(0)
                correct += predicted.eq(targets).sum().item()
                all_preds.extend(predicted.cpu().numpy())
                all_targets.extend(targets.cpu().numpy())
                all_probs.extend(probs.cpu().numpy())

        accuracy = 100. * correct / total
        precision, recall, f1, _ = precision_recall_fscore_support(all_targets, all_preds, average='weighted', zero_division=0)
        kappa = cohen_kappa_score(all_targets, all_preds)
        return {
            'loss': total_loss/len(loader),
            'accuracy': accuracy,
            'precision': precision*100,
            'recall': recall*100,
            'f1': f1*100,
            'kappa': kappa*100
        }

    def train(self):
        print("Starting training...")
        for epoch in range(self.config.epochs):
            self.current_epoch = epoch
            train_metrics = self.train_epoch()
            self.scheduler.step(epoch)
            current_lr = self.optimizer.param_groups[0]['lr']
            val_metrics = self.validate(self.val_loader)

            # Store for plots
            self.train_losses.append(train_metrics['loss'])
            self.train_accs.append(train_metrics['accuracy'])
            self.val_losses.append(val_metrics['loss'])
            self.val_accs.append(val_metrics['accuracy'])

            self.writer.add_scalar('Loss/train', train_metrics['loss'], epoch)
            self.writer.add_scalar('Loss/val', val_metrics['loss'], epoch)
            self.writer.add_scalar('Acc/train', train_metrics['accuracy'], epoch)
            self.writer.add_scalar('Acc/val', val_metrics['accuracy'], epoch)
            self.writer.add_scalar('LR', current_lr, epoch)

            print(f"\nEpoch {epoch}: Train Loss={train_metrics['loss']:.4f} Acc={train_metrics['accuracy']:.2f}% | "
                  f"Val Loss={val_metrics['loss']:.4f} Acc={val_metrics['accuracy']:.2f}% | LR={current_lr:.6f}")

            if val_metrics['accuracy'] > self.best_val_acc:
                self.best_val_acc = val_metrics['accuracy']
                self.patience_counter = 0
                checkpoint = {
                    'epoch': epoch,
                    'model_state_dict': self.model.state_dict(),
                    'optimizer_state_dict': self.optimizer.state_dict(),
                    'scaler_state_dict': self.scaler.state_dict(),
                    'best_val_acc': self.best_val_acc,
                    'val_metrics': val_metrics,
                    'config': self.config
                }
                torch.save(checkpoint, self.save_dir / 'best_model.pth')
                print(f"New best model saved! Val Acc: {self.best_val_acc:.2f}%")
            else:
                self.patience_counter += 1

            if val_metrics['loss'] < self.best_val_loss:
                self.best_val_loss = val_metrics['loss']
                torch.save(checkpoint, self.save_dir / 'best_loss_model.pth')

            if epoch % 10 == 0:
                torch.save(checkpoint, self.save_dir / f'checkpoint_epoch_{epoch}.pth')

            if self.patience_counter >= self.config.early_stop_patience:
                print(f"Early stopping after {epoch+1} epochs")
                break

        print(f"Training completed. Best val accuracy: {self.best_val_acc:.2f}%")
        self.writer.close()

    def get_predictions(self, loader):
        self.model.eval()
        all_preds, all_targets, all_probs = [], [], []
        with torch.no_grad():
            for images, targets in tqdm(loader, desc="Predicting"):
                images = images.to(self.device)
                outputs = self.model(images)
                probs = F.softmax(outputs, dim=1)
                _, preds = outputs.max(1)
                all_preds.extend(preds.cpu().numpy())
                all_targets.extend(targets.numpy())
                all_probs.extend(probs.cpu().numpy())
        return np.array(all_targets), np.array(all_preds), np.array(all_probs)

    def evaluate_full(self, class_names):
        y_true, y_pred, y_probs = self.get_predictions(self.test_loader)
        out_dir = self.save_dir / 'final_evaluation'
        out_dir.mkdir(exist_ok=True)

        # Basic metrics
        acc = accuracy_score(y_true, y_pred)
        prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted')
        kappa = cohen_kappa_score(y_true, y_pred)

        print(f"\n=== FINAL TEST METRICS ===")
        print(f"Accuracy:  {acc:.4f}")
        print(f"Precision: {prec:.4f}")
        print(f"Recall:    {rec:.4f}")
        print(f"F1:        {f1:.4f}")
        print(f"Kappa:     {kappa:.4f}")

        # Extra metrics
        bal_acc = balanced_accuracy_score(y_true, y_pred)
        mcc = matthews_corrcoef(y_true, y_pred)
        cm = confusion_matrix(y_true, y_pred)
        tn = cm.sum() - (cm.sum(axis=0) + cm.sum(axis=1) - np.diag(cm))
        fp = cm.sum(axis=0) - np.diag(cm)
        spec = np.mean(tn / (tn + fp + 1e-8))
        failure = 1 - acc
        max_probs = np.max(y_probs, axis=1)
        wrong = (y_pred != y_true)
        hcf = np.sum(wrong & (max_probs > 0.95)) / len(y_true)
        print(f"Balanced Acc: {bal_acc:.4f}, MCC: {mcc:.4f}, Specificity: {spec:.4f}")
        print(f"Failure Rate: {failure:.4f}, High-Conf Failures: {hcf:.4f}")

        # Class-wise metrics (4 decimals)
        per_prec, per_rec, per_f1, _ = precision_recall_fscore_support(y_true, y_pred, average=None, zero_division=0)
        per_acc = cm.diagonal() / (cm.sum(axis=1) + 1e-8)
        class_df = pd.DataFrame({
            'Class': class_names,
            'Accuracy': per_acc,
            'Precision': per_prec,
            'Recall': per_rec,
            'F1': per_f1
        })
        class_df.to_csv(out_dir/'class_wise_metrics.csv', index=False, float_format='%.4f')

        # Confusion matrix (dynamic size, 600 DPI)
        n_classes = len(class_names)
        figsize = (max(10, n_classes*0.5), max(8, n_classes*0.4))
        plt.figure(figsize=figsize)
        cm_norm = cm.astype('float') / (cm.sum(axis=1)[:, np.newaxis] + 1e-8)
        sns.heatmap(cm_norm, annot=False, fmt='.2f', cmap='Blues',
                    xticklabels=class_names, yticklabels=class_names,
                    cbar_kws={'label': 'Proportion'})
        plt.title('Confusion Matrix', fontweight='bold')
        plt.xlabel('Predicted', fontweight='bold')
        plt.ylabel('True', fontweight='bold')
        plt.xticks(rotation=90, fontsize=8, weight='bold')
        plt.yticks(rotation=0, fontsize=8, weight='bold')
        plt.tight_layout()
        plt.savefig(out_dir/'confusion_matrix.png', dpi=600, bbox_inches='tight')
        plt.close()

        # ROC and PR curves
        self._plot_roc_pr(y_true, y_probs, class_names, out_dir)

        # Extra plots
        self._plot_training_curves(out_dir/'training_curves.png')
        self._plot_calibration_curve(y_true, y_probs, out_dir/'calibration_curve.png')
        self._plot_misclassification(y_true, y_pred, class_names, out_dir/'misclassification_top10.png')
        self._plot_top_confused_pairs(y_true, y_pred, class_names, out_dir/'confused_pairs.png')

        # Model complexity
        params, flops, inf_time = self._compute_complexity()
        with open(out_dir/'complexity.txt', 'w') as f:
            f.write(f"Trainable parameters: {params}\n")
            if flops: f.write(f"FLOPs (G): {flops:.3f}\n")
            f.write(f"Inference time (ms per image): {inf_time:.2f}\n")

        return {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1, 'kappa': kappa}

    def _plot_roc_pr(self, y_true, y_probs, class_names, out_dir):
        from sklearn.metrics import average_precision_score, precision_recall_curve
        n_classes = len(class_names)
        # ROC
        plt.figure(figsize=(12,10))
        fpr, tpr, roc_auc = {}, {}, {}
        for i in range(n_classes):
            fpr[i], tpr[i], _ = roc_curve((y_true==i).astype(int), y_probs[:,i])
            roc_auc[i] = auc(fpr[i], tpr[i])
        all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
        mean_tpr = np.zeros_like(all_fpr)
        for i in range(n_classes):
            mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
        mean_tpr /= n_classes
        plt.plot(all_fpr, mean_tpr, 'b-', label=f'Macro-average ROC (AUC={auc(all_fpr, mean_tpr):.3f})')
        plt.plot([0,1],[0,1],'k--')
        plt.xlabel('False Positive Rate', fontweight='bold')
        plt.ylabel('True Positive Rate', fontweight='bold')
        plt.title('ROC Curves', fontweight='bold')
        plt.legend(loc='upper left', bbox_to_anchor=(1.0,1.0))
        plt.tight_layout(rect=[0,0,0.85,1])
        plt.savefig(out_dir/'roc_curves.png', dpi=600, bbox_inches='tight')
        plt.close()
        # PRC
        plt.figure(figsize=(12,10))
        for i in range(n_classes):
            prec, rec, _ = precision_recall_curve((y_true==i).astype(int), y_probs[:,i])
            ap = average_precision_score((y_true==i).astype(int), y_probs[:,i])
            plt.plot(rec, prec, linewidth=1, label=f'{class_names[i][:15]} (AP={ap:.2f})')
        plt.xlabel('Recall', fontweight='bold')
        plt.ylabel('Precision', fontweight='bold')
        plt.title('Precision-Recall Curves', fontweight='bold')
        plt.legend(loc='upper left', bbox_to_anchor=(1.0,1.0), fontsize=6)
        plt.tight_layout(rect=[0,0,0.85,1])
        plt.savefig(out_dir/'pr_curves.png', dpi=600, bbox_inches='tight')
        plt.close()

    def _plot_training_curves(self, save_path):
        epochs = range(1, len(self.train_losses)+1)
        fig, (ax1, ax2) = plt.subplots(1,2, figsize=(15,5))
        ax1.plot(epochs, self.train_losses, 'b-', label='Train')
        ax1.plot(epochs, self.val_losses, 'r-', label='Val')
        ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.legend(); ax1.grid(True)
        ax2.plot(epochs, self.train_accs, 'b-', label='Train')
        ax2.plot(epochs, self.val_accs, 'r-', label='Val')
        ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.legend(); ax2.grid(True)
        plt.suptitle('Training Curves', fontweight='bold')
        plt.tight_layout()
        plt.savefig(save_path, dpi=600)
        plt.close()

    def _plot_calibration_curve(self, y_true, y_probs, save_path):
        max_probs = np.max(y_probs, axis=1)
        correct = (np.argmax(y_probs, axis=1) == y_true).astype(int)
        prob_true, prob_pred = calibration_curve(correct, max_probs, n_bins=10)
        plt.figure(figsize=(8,8))
        plt.plot(prob_pred, prob_true, marker='o', linewidth=2, label='Model')
        plt.plot([0,1],[0,1], 'k--', label='Perfect')
        plt.xlabel('Mean Predicted Probability', fontweight='bold')
        plt.ylabel('Fraction of Positives', fontweight='bold')
        plt.title('Calibration Curve', fontweight='bold')
        plt.legend(); plt.grid(True)
        plt.tight_layout()
        plt.savefig(save_path, dpi=600, bbox_inches='tight')
        plt.close()

    def _plot_misclassification(self, y_true, y_pred, class_names, save_path, top_k=10):
        cm = confusion_matrix(y_true, y_pred)
        errors = cm.sum(axis=1) - np.diag(cm)
        idx = np.argsort(errors)[-top_k:][::-1]
        plt.figure(figsize=(12,6))
        plt.barh([class_names[i] for i in idx], errors[idx], color='crimson')
        plt.xlabel('Number of Misclassifications', fontweight='bold')
        plt.title(f'Top {top_k} Most Misclassified Classes', fontweight='bold')
        plt.tight_layout()
        plt.savefig(save_path, dpi=600, bbox_inches='tight')
        plt.close()

    def _plot_top_confused_pairs(self, y_true, y_pred, class_names, save_path, top_k=5):
        cm = confusion_matrix(y_true, y_pred)
        n = len(class_names)
        pairs = [((i,j), cm[i,j]) for i in range(n) for j in range(n) if i!=j]
        pairs.sort(key=lambda x: x[1], reverse=True)
        top = pairs[:top_k]
        labels = [f"{class_names[i]} -> {class_names[j]}" for (i,j),_ in top]
        counts = [c for _,c in top]
        plt.figure(figsize=(10,6))
        plt.barh(labels, counts, color='darkorange')
        plt.xlabel('Number of Confusions', fontweight='bold')
        plt.title(f'Top {top_k} Most Confused Class Pairs', fontweight='bold')
        plt.tight_layout()
        plt.savefig(save_path, dpi=600, bbox_inches='tight')
        plt.close()

    def _compute_complexity(self):
        self.model.eval()
        params = count_parameters(self.model)
        flops = None
        if THOP_AVAILABLE:
            dummy = torch.randn(1,3,self.config.input_size,self.config.input_size).to(self.device)
            flops, _ = profile(self.model, inputs=(dummy,), verbose=False)
            flops = flops / 1e9
        dummy = torch.randn(1,3,self.config.input_size,self.config.input_size).to(self.device)
        torch.cuda.synchronize()
        start = time.time()
        for _ in range(100):
            _ = self.model(dummy)
        torch.cuda.synchronize()
        inf_time = (time.time() - start) / 100 * 1000
        return params, flops, inf_time

## Cell 8: Main Training Script (Single Dataset)

In [8]:
def main():
    config = Config()
    config.print_config()
    device = torch.device(config.device)
    print(f"Using device: {device}")
    set_seed(config.seed)

    print("\nLoading datasets...")
    if not os.path.exists(config.dataset_root):
        print(f"ERROR: Dataset path '{config.dataset_root}' does not exist!")
        sys.exit(1)

    train_dataset = NWPURESISC45Dataset(config.dataset_root, split='train', transform=get_train_transforms(config.input_size))
    test_dataset_full = NWPURESISC45Dataset(config.dataset_root, split='test', transform=get_test_transforms(config.input_size))

    val_size = len(test_dataset_full) // 2
    test_size = len(test_dataset_full) - val_size
    val_dataset, test_dataset = random_split(test_dataset_full, [val_size, test_size],
                                             generator=torch.Generator().manual_seed(config.seed))

    print(f"Train: {len(train_dataset)} images")
    print(f"Validation: {len(val_dataset)} images")
    print(f"Test: {len(test_dataset)} images")

    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True,
                              num_workers=config.num_workers, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False,
                            num_workers=config.num_workers, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False,
                             num_workers=config.num_workers, pin_memory=True)

    # Use full model (all flags True)
    model = RemoteSensingHybridCNN(
        num_classes=config.num_classes,
        init_channels=config.init_channels,
        drop_path_rate=config.drop_path_rate,
        dropout_rate=config.dropout_rate,
        use_se=True, use_cbam=False, use_self_attn=True, use_aspp=True
    )

    print(f"Model parameters: {count_parameters(model):,}")
    if THOP_AVAILABLE:
        dummy = torch.randn(1,3,config.input_size,config.input_size)
        flops, _ = profile(model, inputs=(dummy,), verbose=False)
        flops, params = clever_format([flops, count_parameters(model)], "%.3f")
        print(f"FLOPs: {flops}, Parameters: {params}")

    trainer = Trainer(model, train_loader, val_loader, test_loader, config.num_classes,
                      device, config.save_dir, config)
    trainer.train()

    # Load best model for evaluation
    checkpoint_path = Path(config.save_dir) / 'best_model.pth'
    if checkpoint_path.exists():
        checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)  # FIXED
        model.load_state_dict(checkpoint['model_state_dict'])
        print(f"Loaded best model from epoch {checkpoint['epoch']} with val acc {checkpoint['best_val_acc']:.2f}%")
    else:
        print("Best model not found, using current model.")

    # Full evaluation
    results = trainer.evaluate_full(train_dataset.classes)
    print("\n" + "="*60)
    print("FINAL TEST RESULTS")
    print("="*60)
    print(f"Test Accuracy:  {results['accuracy']*100:.2f}%")
    print(f"Test Precision: {results['precision']*100:.2f}%")
    print(f"Test Recall:    {results['recall']*100:.2f}%")
    print(f"Test F1:        {results['f1']*100:.2f}%")
    print(f"Test Kappa:     {results['kappa']:.2f}%")
    print("="*60)

if __name__ == '__main__':
    main()


CONFIGURATION
dataset_name: NWPU
dataset_roots: {'NWPU': 'C:\\Users\\aipmu\\Videos\\Lesss go\\Datasets_Prepared\\NWPU', 'EuroSAT': 'C:\\Users\\aipmu\\Videos\\Lesss go\\Datasets_Prepared\\EuroSAT', 'PatternNet': 'C:\\Users\\aipmu\\Videos\\Lesss go\\Datasets_Prepared\\PatternNet', 'MLRSNet': 'C:\\Users\\aipmu\\Videos\\Lesss go\\Datasets_Prepared\\MLRSNet', 'AID': 'C:\\Users\\aipmu\\Videos\\Lesss go\\Datasets_Prepared\\AID'}
num_classes_dict: {'NWPU': 45, 'EuroSAT': 10, 'PatternNet': 38, 'MLRSNet': 46, 'AID': 30}
dataset_root: <property object at 0x000001A8665AFD80>
num_classes: <property object at 0x000001A8665CEED0>
input_size: 256
init_channels: 16
batch_size: 64
num_workers: 0
epochs: 150
lr: 0.001
weight_decay: 0.01
betas: (0.9, 0.999)
warmup_epochs: 5
min_lr: 1e-06
label_smoothing: 0.1
mixup_alpha: 0.2
cutmix_alpha: 0.2
drop_path_rate: 0.1
dropout_rate: 0.3
seed: 42
device: cuda
save_dir: nwpu_resisc45_checkpoints
early_stop_patience: 20
print_config: <classmethod(<function Config.

Validation: 100%|██████████| 36/36 [00:06<00:00,  5.66it/s]



Epoch 0: Train Loss=3.4797 Acc=9.84% | Val Loss=3.1840 Acc=16.31% | LR=0.000000
New best model saved! Val Acc: 16.31%


Validation: 100%|██████████| 36/36 [00:06<00:00,  5.68it/s]



Epoch 1: Train Loss=3.2988 Acc=14.46% | Val Loss=3.0567 Acc=18.40% | LR=0.000200
New best model saved! Val Acc: 18.40%


...


Epoch 109: Train Loss=1.4098 Acc=79.55% | Val Loss=0.9320 Acc=93.16% | LR=0.000922


Validation: 100%|██████████| 36/36 [00:05<00:00,  6.68it/s]



Epoch 110: Train Loss=1.3430 Acc=81.77% | Val Loss=0.9359 Acc=92.58% | LR=0.000905
Early stopping after 111 epochs
Training completed. Best val accuracy: 95.24%
Loaded best model from epoch 90 with val acc 95.24%


Predicting: 100%|██████████| 36/36 [00:07<00:00,  4.88it/s]



=== FINAL TEST METRICS ===
Accuracy:  0.9462
Precision: 0.9473
Recall:    0.9462
F1:        0.9461
Kappa:     0.9450
Balanced Acc: 0.9460, MCC: 0.9450, Specificity: 0.9988
Failure Rate: 0.0538, High-Conf Failures: 0.0000

FINAL TEST RESULTS
Test Accuracy:  94.62%
Test Precision: 94.73%
Test Recall:    94.62%
Test F1:        94.61%
Test Kappa:     0.94%


## Cell 9: Ablation Study (All Datasets)

In [ ]:
import os

all_results = []
# Load existing results if any
if os.path.exists("ablation_results.csv"):
    existing = pd.read_csv("ablation_results.csv")
    all_results = existing.to_dict('records')
    print(f"Loaded {len(all_results)} existing results")

ablation_configs = [
    {"name": "Baseline",     "use_se": False, "use_cbam": False, "use_self_attn": False, "use_aspp": False, "dropout_rate": 0.0, "drop_path_rate": 0.0},
    {"name": "+ SE",         "use_se": True,  "use_cbam": False, "use_self_attn": False, "use_aspp": False, "dropout_rate": 0.3, "drop_path_rate": 0.1},
    {"name": "+ SpatialAttn","use_se": False, "use_cbam": True,  "use_self_attn": False, "use_aspp": False, "dropout_rate": 0.3, "drop_path_rate": 0.1},
    {"name": "+ CBAM",       "use_se": True,  "use_cbam": True,  "use_self_attn": False, "use_aspp": False, "dropout_rate": 0.3, "drop_path_rate": 0.1},
    {"name": "+ SelfAttn",   "use_se": False, "use_cbam": False, "use_self_attn": True,  "use_aspp": False, "dropout_rate": 0.3, "drop_path_rate": 0.1},
    {"name": "+ ASPP",       "use_se": False, "use_cbam": False, "use_self_attn": False, "use_aspp": True,  "dropout_rate": 0.3, "drop_path_rate": 0.1},
    {"name": "Full Model",   "use_se": True,  "use_cbam": True,  "use_self_attn": True,  "use_aspp": True,  "dropout_rate": 0.3, "drop_path_rate": 0.1},
]

datasets = ["NWPU", "EuroSAT", "PatternNet", "MLRSNet", "AID"]

def train_resumable(trainer, start_epoch, config):
    print("Starting training...")
    for epoch in range(start_epoch, config.epochs):
        trainer.current_epoch = epoch
        train_metrics = trainer.train_epoch()
        trainer.scheduler.step(epoch)
        current_lr = trainer.optimizer.param_groups[0]['lr']
        val_metrics = trainer.validate(trainer.val_loader)

        trainer.train_losses.append(train_metrics['loss'])
        trainer.train_accs.append(train_metrics['accuracy'])
        trainer.val_losses.append(val_metrics['loss'])
        trainer.val_accs.append(val_metrics['accuracy'])

        print(f"\nEpoch {epoch}: Train Loss={train_metrics['loss']:.4f} Acc={train_metrics['accuracy']:.2f}% | "
              f"Val Loss={val_metrics['loss']:.4f} Acc={val_metrics['accuracy']:.2f}% | LR={current_lr:.6f}")

        if val_metrics['accuracy'] > trainer.best_val_acc:
            trainer.best_val_acc = val_metrics['accuracy']
            trainer.patience_counter = 0
            checkpoint = {
                'epoch': epoch,
                'model_state_dict': trainer.model.state_dict(),
                'optimizer_state_dict': trainer.optimizer.state_dict(),
                'scaler_state_dict': trainer.scaler.state_dict(),
                'best_val_acc': trainer.best_val_acc,
                'val_metrics': val_metrics,
                'config': config
            }
            torch.save(checkpoint, trainer.save_dir / 'best_model.pth')
            print(f"New best model saved! Val Acc: {trainer.best_val_acc:.2f}%")
        else:
            trainer.patience_counter += 1

        if val_metrics['loss'] < trainer.best_val_loss:
            trainer.best_val_loss = val_metrics['loss']
            torch.save(checkpoint, trainer.save_dir / 'best_loss_model.pth')

        if epoch % 10 == 0:
            torch.save(checkpoint, trainer.save_dir / f'checkpoint_epoch_{epoch}.pth')

        if trainer.patience_counter >= config.early_stop_patience:
            print(f"Early stopping at epoch {epoch}")
            break

    print(f"Done. Best val acc: {trainer.best_val_acc:.2f}%")
    trainer.writer.close()

for dname in datasets:
    config = Config()
    config.dataset_name = dname
    set_seed(config.seed)
    print(f"\n{'='*60}\nDataset: {dname}\n{'='*60}")

    if not os.path.exists(config.dataset_root):
        print(f"ERROR: {config.dataset_root} not found, skipping.")
        continue

    # Load data
    train_ds = NWPURESISC45Dataset(config.dataset_root, 'train', get_train_transforms(config.input_size))
    test_ds  = NWPURESISC45Dataset(config.dataset_root, 'test',  get_test_transforms(config.input_size))
    val_size = len(test_ds) // 2
    val_ds, test_ds = random_split(test_ds, [val_size, len(test_ds) - val_size],
                                   generator=torch.Generator().manual_seed(config.seed))
    train_loader = DataLoader(train_ds, config.batch_size, shuffle=True,
                              num_workers=config.num_workers, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds,   config.batch_size, shuffle=False,
                              num_workers=config.num_workers, pin_memory=True)

    for cfg in ablation_configs:
        save_dir = f"ablation_{dname}_{cfg['name'].replace(' ', '_').replace('+', 'plus')}"

        # ── SKIP if already completed ──────────────────────────────────────────
        already_done = any(
            r['Dataset'] == dname and r['Experiment'] == cfg['name']
            for r in all_results
        )
        if already_done:
            print(f"⏭️  Skipping {dname} - {cfg['name']} (already in CSV)")
            continue

        # ── CHECK for existing checkpoint ──────────────────────────────────────
        checkpoint_path = Path(save_dir) / 'best_model.pth'
        resume_epoch = 0
        resume_checkpoint = None
        if checkpoint_path.exists():
            resume_checkpoint = torch.load(checkpoint_path, map_location=config.device, weights_only=False)
            resume_epoch = resume_checkpoint['epoch'] + 1
            print(f"\n--- {cfg['name']} (resuming from epoch {resume_epoch}) ---")
        else:
            print(f"\n--- {cfg['name']} (fresh start) ---")

        # ── BUILD MODEL ────────────────────────────────────────────────────────
        model = RemoteSensingHybridCNN(
            num_classes=config.num_classes,
            init_channels=config.init_channels,
            drop_path_rate=cfg['drop_path_rate'],
            dropout_rate=cfg['dropout_rate'],
            use_se=cfg['use_se'],
            use_cbam=cfg['use_cbam'],
            use_self_attn=cfg['use_self_attn'],
            use_aspp=cfg['use_aspp']
        )

        trainer = Trainer(model, train_loader, val_loader, val_loader,
                          config.num_classes, config.device, save_dir, config)

        # ── RESTORE STATE if resuming ──────────────────────────────────────────
        if resume_checkpoint is not None:
            model.load_state_dict(resume_checkpoint['model_state_dict'])
            trainer.optimizer.load_state_dict(resume_checkpoint['optimizer_state_dict'])
            trainer.scaler.load_state_dict(resume_checkpoint['scaler_state_dict'])
            trainer.best_val_acc  = resume_checkpoint['best_val_acc']
            trainer.best_val_loss = float('inf')  # conservative reset
            trainer.patience_counter = 0          # reset patience — we don't know the old counter
            trainer.current_epoch = resume_epoch

            # Fast-forward scheduler to correct LR position
            for e in range(resume_epoch):
                trainer.scheduler.step(e)

            print(f"   Restored best val acc : {trainer.best_val_acc:.2f}%")
            print(f"   Scheduler fast-forwarded to epoch {resume_epoch}")
            print(f"   Current LR             : {trainer.optimizer.param_groups[0]['lr']:.6f}")

        # ── TRAIN ──────────────────────────────────────────────────────────────
        train_resumable(trainer, resume_epoch, config)

        # ── SAVE RESULT ────────────────────────────────────────────────────────
        val_acc = trainer.best_val_acc
        all_results.append({"Dataset": dname, "Experiment": cfg["name"], "Val_Accuracy": val_acc})
        pd.DataFrame(all_results).to_csv("ablation_results.csv", index=False)
        print(f"✅ {dname} - {cfg['name']}: Val Acc = {val_acc:.2f}%")

print("\n🎉 Ablation finished. Results saved to ablation_results.csv")

Loaded 4 existing results

Dataset: NWPU
Loaded train split: 27000 images, 45 classes
Loaded test split: 4500 images, 45 classes
⏭️  Skipping NWPU - Baseline (already in CSV)
⏭️  Skipping NWPU - + SE (already in CSV)
⏭️  Skipping NWPU - + SpatialAttn (already in CSV)
⏭️  Skipping NWPU - + CBAM (already in CSV)

--- + SelfAttn (fresh start) ---
Model has 13,971,741 trainable parameters
Starting training...


Epoch 0:   0%|          | 0/421 [00:00<?, ?it/s]

## Cell 10: 5-Fold Cross-Validation on One Dataset

In [ ]:
from sklearn.model_selection import StratifiedKFold

def cross_validate(dataset_name, config_dict, n_folds=5):
    config = Config()
    config.dataset_name = dataset_name
    set_seed(config.seed)
    full_ds = NWPURESISC45Dataset(config.dataset_root, 'train', get_train_transforms(config.input_size))
    labels = [label for _, label in full_ds]
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=config.seed)

    fold_accs = []
    all_true, all_pred = [], []

    for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(labels)), labels)):
        print(f"\nFold {fold+1}/{n_folds}")
        train_sub = torch.utils.data.Subset(full_ds, train_idx)
        val_sub = torch.utils.data.Subset(full_ds, val_idx)
        train_loader = DataLoader(train_sub, config.batch_size, shuffle=True, num_workers=config.num_workers)
        val_loader = DataLoader(val_sub, config.batch_size, shuffle=False, num_workers=config.num_workers)

        model = RemoteSensingHybridCNN(
            num_classes=config.num_classes,
            init_channels=config.init_channels,
            drop_path_rate=config.drop_path_rate,
            dropout_rate=config.dropout_rate,
            use_se=True, use_cbam=True, use_self_attn=True, use_aspp=True
        )
        trainer = Trainer(model, train_loader, val_loader, val_loader, config.num_classes,
                          config.device, f"cv_{dataset_name}_fold{fold+1}", config)
        trainer.train()
        y_true, y_pred, _ = trainer.get_predictions(val_loader)
        acc = accuracy_score(y_true, y_pred)
        fold_accs.append(acc)
        all_true.extend(y_true); all_pred.extend(y_pred)
        print(f"Fold {fold+1} accuracy: {acc:.4f}")

    mean_acc = np.mean(fold_accs)
    std_acc = np.std(fold_accs)
    print(f"\nCV Accuracy: {mean_acc:.4f} ± {std_acc:.4f}")

    out_dir = Path(f"cv_{dataset_name}")
    out_dir.mkdir(exist_ok=True)
    # Overall confusion matrix
    cm = confusion_matrix(all_true, all_pred)
    n_classes = len(full_ds.classes)
    figsize = (max(10, n_classes*0.5), max(8, n_classes*0.4))
    plt.figure(figsize=figsize)
    sns.heatmap(cm, annot=False, fmt='d', cmap='Blues',
                xticklabels=full_ds.classes, yticklabels=full_ds.classes,
                cbar_kws={'label': 'Count'})
    plt.title(f'{dataset_name} - {n_folds}-Fold CV Confusion Matrix', fontweight='bold')
    plt.xlabel('Predicted', fontweight='bold')
    plt.ylabel('True', fontweight='bold')
    plt.xticks(rotation=90, fontsize=8, weight='bold')
    plt.yticks(rotation=0, fontsize=8, weight='bold')
    plt.tight_layout()
    plt.savefig(out_dir/'cv_confusion_matrix.png', dpi=600, bbox_inches='tight')
    plt.close()

    report = classification_report(all_true, all_pred, target_names=full_ds.classes, digits=4)
    with open(out_dir/'cv_classification_report.txt', 'w') as f:
        f.write(report)
    return mean_acc, std_acc

# Example: Run CV for EuroSAT (fast)
mean_acc, std_acc = cross_validate("EuroSAT", Config, n_folds=5)
print(f"5-Fold CV Accuracy: {mean_acc*100:.2f}% ± {std_acc*100:.2f}%")

## Cell 11: Final Evaluation on Best Model (Full Model) for All Datasets

In [ ]:
# After ablation, run final evaluation with full model on each dataset
best_config = ablation_configs[-1]  # Full Model
datasets = ["NWPU", "EuroSAT", "PatternNet", "MLRSNet", "AID"]

for dname in datasets:
    config = Config()
    config.dataset_name = dname
    config.save_dir = f"final_{dname}"
    set_seed(config.seed)

    train_ds = NWPURESISC45Dataset(config.dataset_root, 'train', get_train_transforms(config.input_size))
    test_ds = NWPURESISC45Dataset(config.dataset_root, 'test', get_test_transforms(config.input_size))
    train_loader = DataLoader(train_ds, config.batch_size, shuffle=True, num_workers=config.num_workers, pin_memory=True)
    test_loader = DataLoader(test_ds, config.batch_size, shuffle=False, num_workers=config.num_workers, pin_memory=True)

    model = RemoteSensingHybridCNN(
        num_classes=config.num_classes,
        init_channels=config.init_channels,
        drop_path_rate=best_config['drop_path_rate'],
        dropout_rate=best_config['dropout_rate'],
        use_se=best_config['use_se'], use_cbam=best_config['use_cbam'],
        use_self_attn=best_config['use_self_attn'], use_aspp=best_config['use_aspp']
    )
    trainer = Trainer(model, train_loader, test_loader, test_loader, config.num_classes,
                      config.device, config.save_dir, config)
    trainer.train()
    results = trainer.evaluate_full(train_ds.classes)
    print(f"\nFinal results for {dname}: Accuracy = {results['accuracy']*100:.2f}%\n")